# Muddy children — an epistemic puzzle, recovered exactly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sulcantonin/torchmodal/blob/main/examples/notebooks/01_muddy_children.ipynb)

Three children are playing; all three get mud on their foreheads. Each can see the
others but not themselves. Their father announces **"at least one of you is muddy"**,
then asks repeatedly whether anyone knows their own state.

Nobody knows after the first question, nor after the second — but after the **third**,
all three suddenly do. Nothing new is ever observed: the only information that moves
them is the fact that nobody else could answer.

This notebook encodes the puzzle as a Kripke model and evaluates `K_a(muddy_a)` with
torchmodal's necessity neuron, checking that the learned interval **brackets** the
crisp textbook answer at every round.


In [ ]:
# Colab: install the library. Locally, this is a no-op if it is already present.
try:
    import torchmodal
except ImportError:
    !pip install -q torchmodal
    import torchmodal

print('torchmodal', torchmodal.__version__)


In [ ]:
import torch
from itertools import product
from torchmodal import functional as F

N, TAU = 3, 0.05
WORLDS = list(product([0, 1], repeat=N))   # one bit per child, 1 = muddy
ACTUAL = (1, 1, 1)                          # every child really is muddy
here = WORLDS.index(ACTUAL)
print(f'{len(WORLDS)} possible worlds; the actual one is {ACTUAL}')


## The accessibility relation

Two worlds are **indistinguishable** to child *a* when they agree on every *other*
child. That makes each child's relation an equivalence — the standard S5 frame for
knowledge.


In [ ]:
def indistinguishable(agent):
    n = len(WORLDS)
    A = torch.zeros(n, n)
    for i, w in enumerate(WORLDS):
        for j, v in enumerate(WORLDS):
            if all(w[k] == v[k] for k in range(N) if k != agent):
                A[i, j] = 1.0
    return A

indistinguishable(0)


## Announcements remove worlds

The father's announcement kills the all-clean world. Each later round is the public
fact that *nobody knew yet*, which kills every world with exactly that many muddy
children. We apply this by zeroing the accessibility **into** the removed worlds.


In [ ]:
def surviving(rnd):
    return torch.tensor([float(sum(w) > rnd) for w in WORLDS])

def knows(agent, rnd):
    A = indistinguishable(agent) * surviving(rnd).unsqueeze(0)
    muddy = torch.tensor([[float(w[agent])] * 2 for w in WORLDS])
    return F.necessity(muddy, A, tau=TAU)[here]

def crisp(agent, rnd):
    keep = surviving(rnd)
    considered = [v for j, v in enumerate(WORLDS) if keep[j] > 0
                  and all(ACTUAL[k] == v[k] for k in range(N) if k != agent)]
    return int(all(v[agent] == 1 for v in considered))


## The result


In [ ]:
print(f"{'after round':>12}  {'child':>5}  {'crisp':>5}   learned bounds")
print('-' * 56)
for rnd in range(N):
    for agent in range(N):
        L, U = knows(agent, rnd)
        c = crisp(agent, rnd)
        assert L - 1e-4 <= c <= U + 1e-4, 'bounds must contain the crisp value'
        print(f'{rnd + 1:>12}  {agent:>5}  {c:>5}   [{L:.4f}, {U:.4f}]')
    print()
print('Every interval bracketed the crisp answer.')


**No, no, yes** — the textbook result, out of the modal operator alone, with every
interval containing it.


## Soundness is a property you can measure

The interval is not vague: its width is *exactly* `tau * H(w)`, the entropy of the
necessity neuron's own softmin weights. `box_width_entropy` returns it.


In [ ]:
A = indistinguishable(0) * surviving(2).unsqueeze(0)
muddy = torch.tensor([[float(w[0])] * 2 for w in WORLDS])

for tau in (0.3, 0.1, 0.05, 0.01):
    L, U = F.necessity(muddy, A, tau=tau)[here]
    w = F.box_width_entropy(A, muddy, tau=tau)[here]
    print(f'tau={tau:<5}  bounds=[{L:.4f}, {U:.4f}]  '
          f'measured width={U - L:.4f}  box_width_entropy={w:.4f}')


As `tau -> 0` the interval collapses onto the crisp answer, and the predicted width
tracks the measured one. That is the whole soundness story in one table.
